In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
%cd /content/drive/MyDrive/DL/Transformer from Scratch/RNNFinetune

/content/drive/MyDrive/DL/Transformer from Scratch/RNNFinetune


In [19]:
!pip install langdetect

In [20]:
from datasets import load_dataset, Dataset
import pandas as pd
import re
import os
from langdetect import detect, LangDetectException

In [21]:
DATA_ROOT = "/content/drive/MyDrive/DL/Transformer from Scratch/Data_Medi_2"
CSV_PATH = os.path.join(DATA_ROOT, "full_clean_data.csv")

In [22]:
# Tải Dataset gốc
print("Đang tải dataset...")
ds_raw = load_dataset("THPBi/medical-o1-reasoning-SFT-vi-translated", split="train")

# Chuyển sang Pandas DataFrame để xử lý
df = ds_raw.to_pandas()
print(f"Số lượng mẫu gốc: {len(df)}")

Đang tải dataset...
Số lượng mẫu gốc: 25371


In [23]:
# Tách và Gộp cột
df_q = df[['Question', 'Question_Vi']].rename(columns={'Question': 'Eng', 'Question_Vi': 'Vie'})
df_r = df[['Response', 'Response_Vi']].rename(columns={'Response': 'Eng', 'Response_Vi': 'Vie'})

df_final = pd.concat([df_q, df_r], ignore_index=True)
print(f"Số lượng mẫu sau khi gộp Question & Response: {len(df_final)}")

Số lượng mẫu sau khi gộp Question & Response: 50742


In [24]:
# =====================================================
# INLINE MedEV PROCESSOR (GIỐNG medev_processor.py)
# =====================================================
import urllib.request
from pathlib import Path

MED_EV_DIR = "./_medev_tmp"
Path(MED_EV_DIR).mkdir(exist_ok=True)

BASE_URL = "https://huggingface.co/datasets/nhuvo/MedEV/resolve/main/"

FILES = [
    ("train.en.txt", "train_en.txt"),
    ("train.vi.txt", "train_vi.txt"),
    ("val.en.new.txt", "val_en.txt"),
    ("val.vi.new.txt", "val_vi.txt"),
    ("test.en.new.txt", "test_en.txt"),
    ("test.vi.new.txt", "test_vi.txt"),
]

print(">>> Downloading MedEV txt files...")
for remote, local in FILES:
    path = Path(MED_EV_DIR) / local
    if not path.exists():
        urllib.request.urlretrieve(BASE_URL + remote, path)

def load_parallel(en_path, vi_path):
    with open(en_path, "r", encoding="utf-8") as f:
        en_lines = [l.strip() for l in f.readlines()]

    with open(vi_path, "r", encoding="utf-8") as f:
        vi_lines = [l.strip() for l in f.readlines()]

    n = min(len(en_lines), len(vi_lines))
    en_lines = en_lines[:n]
    vi_lines = vi_lines[:n]

    rows = []
    for e, v in zip(en_lines, vi_lines):
        if len(e) == 0 or len(v) == 0:
            continue
        rows.append({"Eng": e, "Vie": v})

    return pd.DataFrame(rows)

print(">>> Processing MedEV splits...")

df_medev_train = load_parallel(
    Path(MED_EV_DIR) / "train_en.txt",
    Path(MED_EV_DIR) / "train_vi.txt"
)

df_medev_val = load_parallel(
    Path(MED_EV_DIR) / "val_en.txt",
    Path(MED_EV_DIR) / "val_vi.txt"
)

df_medev_test = load_parallel(
    Path(MED_EV_DIR) / "test_en.txt",
    Path(MED_EV_DIR) / "test_vi.txt"
)

df_medev = pd.concat(
    [df_medev_train, df_medev_val, df_medev_test],
    ignore_index=True
)

print(f">>> MedEV total samples (raw): {len(df_medev):,}")
print(f">>> df_final before merge: {len(df_final):,}")

df_final = pd.concat([df_final, df_medev], ignore_index=True)

print(f">>> df_final AFTER MedEV merge: {len(df_final):,}")


>>> Downloading MedEV txt files...
>>> Processing MedEV splits...
>>> MedEV total samples (raw): 358,796
>>> df_final before merge: 50,742
>>> df_final AFTER MedEV merge: 409,538


In [25]:
df_final[:5]

,Eng,Vie
0,A 61-year-old woman with a long history of inv...,"vi: """""" Một phụ nữ 61 tuổi với tiền sử mất nướ..."
1,A 45-year-old man with a history of alcohol us...,"vi: """""" Một người đàn ông 45 tuổi có tiền sử s..."
2,A 45-year-old man presents with symptoms inclu...,"vi: """""" Một người đàn ông 45 tuổi có các triệu..."
3,A patient with psoriasis was treated with syst...,"vi: """""" Một bệnh nhân bị vảy nến được điều trị..."
4,What is the most likely diagnosis for a 2-year...,vi: Dịch đoạn văn sau từ tiếng Anh sang tiếng ...


In [26]:
# HÀM CLEAN & DUỖI PHẲNG
def clean_text(text):
    if not isinstance(text, str): return ""

    # Duỗi phẳng dòng
    text = text.replace('\n', ' ').replace('\r', ' ')

    # Xóa prefix rác thường gặp
    text = re.sub(r'^(en|vi|E\.|A\.|B\.|C\.|D\.)\s*[:.-]*\s*', '', text, flags=re.IGNORECASE)

    # Xóa câu lệnh dịch máy thừa thãi
    text = re.sub(r'^(Dịch|Translate|Hãy dịch).*?[:\n]', '', text, flags=re.IGNORECASE)

    # Xóa tag ảnh
    text = re.sub(r'\[img.*?\]', '', text)

    # Xóa nháy
    text = text.strip(' "\'')

    # Xóa khoảng trắng thừa ở giữa câu (nếu có)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

print(">>> Đang clean text (Gọt sạch dấu nháy)...")
df_final['Eng'] = df_final['Eng'].apply(clean_text)
df_final['Vie'] = df_final['Vie'].apply(clean_text)

>>> Đang clean text (Gọt sạch dấu nháy)...


In [27]:
# HÀM LỌC
def is_valid_row(row):
    eng = row['Eng']
    vie = row['Vie']

    # 4.1. Kiểm tra rỗng hoặc quá ngắn
    if not eng or not vie: return False
    if len(eng) < 5 or len(vie) < 5: return False
    if len(eng) > 250: return False

    # 4.2. Kiểm tra ký tự lạ (Tiếng Trung, Hàn, Nhật)
    if re.search(r'[\u4e00-\u9fff]', eng) or re.search(r'[\u4e00-\u9fff]', vie):
        return False

    # 4.3. Kiểm tra Toán học / Code rác (Bắt đầu bằng dấu ngoặc hoặc phép tính)
    if re.match(r'^[\(\)\+\-\*\/\=]', eng):
        return False

    # 4.4. Kiểm tra ngôn ngữ bằng AI (langdetect)
    try:
        if len(vie) > 20:
            lang_vi = detect(vie)
            if lang_vi != 'vi': return False # Cột Việt mà không phải Việt -> Xóa

        if len(eng) > 20:
            lang_en = detect(eng)
            if lang_en != 'en': return False # Cột Anh mà không phải Anh -> Xóa
    except LangDetectException:
        return False

    return True

print(">>> Đang lọc dữ liệu HARDCORE...")
mask = df_final.apply(is_valid_row, axis=1)
df_clean = df_final[mask].reset_index(drop=True)

print(f"Số lượng mẫu ban đầu: {len(df_final)}")
print(f"Số lượng mẫu SAU KHI LỌC: {len(df_clean)}")

>>> Đang lọc dữ liệu HARDCORE...
Số lượng mẫu ban đầu: 409538
Số lượng mẫu SAU KHI LỌC: 331609


In [31]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 331609 entries, 0 to 331608
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   Eng     331609 non-null  object
 1   Vie     331609 non-null  object
dtypes: object(2)
memory usage: 5.1+ MB


In [37]:
df_clean['Eng'].iloc[331606]


'To guideline for the physical activity at the household and at school.'

In [28]:
# Lưu file CSV tổng
if not os.path.exists(DATA_ROOT): os.makedirs(DATA_ROOT)
df_clean.to_csv(CSV_PATH, index=False, encoding='utf-8')
print(f">>> Đã lưu file CSV sạch tại: {CSV_PATH}")

>>> Đã lưu file CSV sạch tại: /content/drive/MyDrive/DL/Transformer from Scratch/Data_Medi_2/full_clean_data.csv


Thêm data mới

https://huggingface.co/datasets/nhuvo/MedEV/viewer/default/train?p=2

In [29]:
# Chia tập Train/Val/Test
print(">>> Đang chia tập Train/Val/Test...")
dataset_clean = Dataset.from_pandas(df_clean)

# Train 90%, Temp 10%
split_1 = dataset_clean.train_test_split(test_size=0.1, seed=42)
temp_ds = split_1['test']

# Temp -> Val 50%, Test 50%
split_2 = temp_ds.train_test_split(test_size=0.5, seed=42)

train_ds = split_1['train']
val_ds = split_2['train']
test_ds = split_2['test']

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

train_ds.save_to_disk(os.path.join(DATA_ROOT, "med_train"))
val_ds.save_to_disk(os.path.join(DATA_ROOT, "med_val"))
test_ds.save_to_disk(os.path.join(DATA_ROOT, "med_test"))

>>> Đang chia tập Train/Val/Test...
Train: 298448 | Val: 16580 | Test: 16581


Saving the dataset (0/1 shards):   0%|          | 0/298448 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/16580 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/16581 [00:00<?, ? examples/s]

In [30]:
df_final[5:]

,Eng,Vie
5,Scientists are developing a new non-steroidal ...,Các nhà khoa học đang phát triển một loại thuố...
6,A 15-year-old boy presents with decreased faci...,"Một cậu bé 15 tuổi biểu hiện với râu giảm, ngự..."
7,In a patient with dermatomyositis as indicated...,Trong một bệnh nhân bị viêm da cơ thể biểu hiệ...
8,Based on the presentation of gait disturbances...,"Dựa trên biểu hiện rối loạn dáng đi, run rẩy, ..."
9,A 25-year-old male presents with high-grade fe...,Một bệnh nhân nam 25 tuổi bị sốt cao và hạ huy...
...,...,...
409533,The pilot intervention was implemented at 4 pr...,Các tác giả xây dựng các giải pháp và tiến hàn...
409534,To setting up the menu for the school children...,Xây dựng các thực đơn cho trẻ TC - BP theo từn...
409535,To guideline for the physical activity at the ...,Hướng dẫn các phương pháp luyện tập thể dục th...
409536,There was a changes of the food habits and hab...,Có sự thay đổi về các tập quán ăn uống và thói...


In [ ]:
dat